In [ ]:
# %% PARAMETERS
# Fabric pipeline parameter
# "" = run all active BRZ_TO_SIL metadata rows
# Example: "SILVER_DIM_ROUTE" = run one Silver target only

p_target_table = ""

# %% CONFIGURATION
# =============================================================================
# TRANSIT - BRONZE -> SILVER TRANSFORMATION + DQ + METADATA DRIVER
# Microsoft Fabric Lakehouse / PySpark
#
# IMPORTANT:
#   1. DDL is intentionally NOT in this notebook/script.
#   2. Run 01_silver_drop_recreate_ddl.sql separately when schema recreation is needed.
#   3. This script performs transformations, DQ logging, Type-1 MERGEs, and fact load.
# =============================================================================

import json
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, TimestampType
)



In [ ]:
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
ADMIN_SCHEMA = "admin_ops"

CONTROL_TABLE = f"{ADMIN_SCHEMA}.admin_load_control"
PROCESSING_LOG_TABLE = f"{ADMIN_SCHEMA}.admin_processing_log"
DQ_TABLE = f"{ADMIN_SCHEMA}.admin_dq_results"

EXECUTION_ID = str(uuid.uuid4())
RUN_TS_DT = datetime.now(timezone.utc)
RUN_TS = RUN_TS_DT.strftime("%Y-%m-%d %H:%M:%S")

# Mapping.xlsx did not define the fiscal year start.
# Keep 1 until business confirms otherwise.
FISCAL_YEAR_START_MONTH = 1

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{SILVER_SCHEMA}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{ADMIN_SCHEMA}`")

for required_table in [CONTROL_TABLE, PROCESSING_LOG_TABLE, DQ_TABLE]:
    if not spark.catalog.tableExists(required_table):
        raise ValueError(
            f"Required table {required_table} does not exist. "
            "Create the admin_ops framework tables before running Silver."
        )

print(f"Silver execution_id={EXECUTION_ID}")
print(f"CONTROL_TABLE={CONTROL_TABLE}")
print(f"PROCESSING_LOG_TABLE={PROCESSING_LOG_TABLE}")
print(f"DQ_TABLE={DQ_TABLE}")

# %% TRANSFORMS / DRIVER

In [ ]:
def _processing_log_id(job_name: str, start_ts: datetime) -> int:
    return hash(
        (EXECUTION_ID, job_name, start_ts.isoformat())
    ) & 0x3FFFFFFFFFFFFFFF


def write_processing_log(
    job_name: str,
    job_start_ts: datetime,
    job_status: str,
    source_table: str,
    destination_table: str,
    total_rows: int = 0,
    information: str = None,
    error_id: str = None,
    error_message: str = None,
    processed_condition: str = None,
    flag: str = "BRZ_TO_SIL",
) -> None:
    job_end_ts = datetime.now(timezone.utc)
    duration_sec = int((job_end_ts - job_start_ts).total_seconds())

    row = [(
        _processing_log_id(job_name, job_start_ts),
        EXECUTION_ID,
        job_name,
        job_start_ts,
        job_end_ts,
        duration_sec,
        job_status,
        information,
        int(total_rows or 0),
        error_id,
        error_message,
        source_table,
        destination_table,
        None,
        None,
        processed_condition,
        flag,
        job_end_ts,
    )]

    schema = StructType([
        StructField("ROW_ID", LongType(), False),
        StructField("EXECUTION_ID", StringType(), False),
        StructField("JOB_NAME", StringType(), False),
        StructField("JOB_START_TS", TimestampType(), False),
        StructField("JOB_END_TS", TimestampType(), False),
        StructField("DURATION_SEC", LongType(), False),
        StructField("JOB_STATUS", StringType(), False),
        StructField("INFORMATION", StringType(), True),
        StructField("TOTAL_ROWS", LongType(), True),
        StructField("ERROR_ID", StringType(), True),
        StructField("ERROR_MESSAGE", StringType(), True),
        StructField("SOURCE_TABLE", StringType(), True),
        StructField("DESTINATION_TABLE", StringType(), True),
        StructField("PULL_FROM_DATE", TimestampType(), True),
        StructField("PULL_TO_DATE", TimestampType(), True),
        StructField("PROCESSED_CONDITION", StringType(), True),
        StructField("FLAG", StringType(), True),
        StructField("INSERT_TS", TimestampType(), False),
    ])

    spark.createDataFrame(row, schema=schema) \
        .write.mode("append").saveAsTable(PROCESSING_LOG_TABLE)


# =============================================================================
# DQ LOGGING
# =============================================================================

def _dq_id(rule_name: str, target_table: str, check_ts: datetime) -> int:
    return hash(
        (EXECUTION_ID, rule_name, target_table, check_ts.isoformat())
    ) & 0x3FFFFFFFFFFFFFFF


def log_dq(
    target_table: str,
    source_tables: str,
    rule_name: str,
    rule_type: str,
    description: str,
    severity: str,
    total_rows: int,
    failed_rows: int,
    sample_details: str = None,
) -> None:
    check_ts = datetime.now(timezone.utc)
    total_rows = int(total_rows or 0)
    failed_rows = int(failed_rows or 0)
    failed_pct = (
        round((failed_rows / total_rows) * 100.0, 6)
        if total_rows > 0
        else 0.0
    )
    status = "PASS" if failed_rows == 0 else "FAIL"

    row = [(
        _dq_id(rule_name, target_table, check_ts),
        EXECUTION_ID,
        "SILVER",
        target_table,
        source_tables,
        rule_name,
        rule_type,
        description,
        severity.upper(),
        status,
        total_rows,
        failed_rows,
        failed_pct,
        sample_details,
        check_ts,
        check_ts,
    )]

    schema = StructType([
        StructField("DQ_RESULT_ID", LongType(), False),
        StructField("EXECUTION_ID", StringType(), False),
        StructField("LAYER_NAME", StringType(), False),
        StructField("TARGET_TABLE", StringType(), False),
        StructField("SOURCE_TABLES", StringType(), True),
        StructField("RULE_NAME", StringType(), False),
        StructField("RULE_TYPE", StringType(), False),
        StructField("RULE_DESCRIPTION", StringType(), True),
        StructField("SEVERITY", StringType(), False),
        StructField("RULE_STATUS", StringType(), False),
        StructField("TOTAL_ROW_COUNT", LongType(), False),
        StructField("FAILED_ROW_COUNT", LongType(), False),
        StructField("FAILED_PCT", DoubleType(), False),
        StructField("SAMPLE_DETAILS", StringType(), True),
        StructField("CHECK_TS", TimestampType(), False),
        StructField("INSERT_TS", TimestampType(), False),
    ])

    spark.createDataFrame(row, schema=schema) \
        .write.mode("append").saveAsTable(DQ_TABLE)


def run_dq(
    df: DataFrame,
    target_table: str,
    source_tables: str,
    rule_name: str,
    rule_type: str,
    description: str,
    severity: str,
    fail_condition,
    sample_columns=None,
    sample_limit: int = 5,
) -> int:
    total_rows = df.count()
    failed_df = df.filter(fail_condition)
    failed_rows = failed_df.count()

    sample_details = None
    if failed_rows > 0:
        cols = sample_columns or df.columns[:8]
        samples = [
            r.asDict(recursive=True)
            for r in failed_df.select(*cols).limit(sample_limit).collect()
        ]
        sample_details = json.dumps(samples, default=str)[:4000]

    log_dq(
        target_table=target_table,
        source_tables=source_tables,
        rule_name=rule_name,
        rule_type=rule_type,
        description=description,
        severity=severity,
        total_rows=total_rows,
        failed_rows=failed_rows,
        sample_details=sample_details,
    )

    if failed_rows > 0 and severity.upper() == "ERROR":
        raise ValueError(
            f"DQ ERROR [{target_table}.{rule_name}] "
            f"{failed_rows:,}/{total_rows:,} row(s) failed"
        )

    if failed_rows > 0:
        print(
            f"DQ WARNING [{target_table}.{rule_name}] "
            f"{failed_rows:,}/{total_rows:,} row(s) failed"
        )

    return failed_rows


def dedupe_latest(df: DataFrame, target_table: str, keys):
    total_rows = df.count()
    order_columns = [
        c for c in ["UPDATED_AT", "UPDATE_TS", "INGESTED_AT", "INSERT_TS", "READ_TS"]
        if c in df.columns
    ]

    if not order_columns:
        order_exprs = [F.monotonically_increasing_id().desc()]
    else:
        order_exprs = [F.col(c).desc_nulls_last() for c in order_columns]

    window_spec = Window.partitionBy(*keys).orderBy(*order_exprs)
    deduped_df = (
        df.withColumn("__silver_dedupe_rank", F.row_number().over(window_spec))
          .filter(F.col("__silver_dedupe_rank") == 1)
          .drop("__silver_dedupe_rank")
    )

    deduped_rows = deduped_df.count()
    duplicate_rows = total_rows - deduped_rows
    print(
        f"{target_table}: source count(*)={total_rows:,}; "
        f"after dedupe={deduped_rows:,}; duplicate rows removed={duplicate_rows:,}"
    )
    return deduped_df

def dq_unique(
    df: DataFrame,
    target_table: str,
    source_tables: str,
    keys,
    severity="WARNING",
):
    total_rows = df.count()
    dup_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
    )
    failed_rows = (
        dup_groups.select(F.sum("count").alias("cnt")).first()["cnt"]
        or 0
    )

    sample_details = None
    if failed_rows:
        samples = [
            r.asDict(recursive=True)
            for r in dup_groups.limit(5).collect()
        ]
        sample_details = json.dumps(samples, default=str)[:4000]

    log_dq(
        target_table=target_table,
        source_tables=source_tables,
        rule_name="UNIQUE_" + "_".join(keys),
        rule_type="UNIQUENESS",
        description=f"Expected unique grain on {keys}",
        severity=severity,
        total_rows=total_rows,
        failed_rows=int(failed_rows),
        sample_details=sample_details,
    )

    if failed_rows and severity.upper() == "ERROR":
        raise ValueError(
            f"DQ ERROR [{target_table}] duplicate grain on {keys}: "
            f"{int(failed_rows):,} row(s)"
        )


# =============================================================================
# TYPE-1 MERGE
# =============================================================================

def merge_type1(
    target_table: str,
    stage_df: DataFrame,
    key_columns,
    compare_columns,
) -> None:
    target_name = f"{SILVER_SCHEMA}.{target_table}"
    stage_df = dedupe_latest(stage_df, target_table, key_columns)

    if not spark.catalog.tableExists(target_name):
        (stage_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_name))
        print(f"{target_table}: created with {stage_df.count():,} row(s)")
        return

    target_columns = spark.table(target_name).columns

    missing = [c for c in target_columns if c not in stage_df.columns]
    if missing:
        raise ValueError(
            f"{target_table}: stage missing target columns: {missing}"
        )

    view_name = f"__stg_{target_table.lower()}_{EXECUTION_ID[:8]}"
    stage_df.select(*target_columns).createOrReplaceTempView(view_name)

    on_expr = " AND ".join(
        [f"t.`{c}` <=> s.`{c}`" for c in key_columns]
    )

    comparison = " AND ".join(
        [f"t.`{c}` <=> s.`{c}`" for c in compare_columns]
    )
    changed = f"NOT ({comparison})" if comparison else "false"

    update_parts = []
    for c in target_columns:
        if c in key_columns or c == "INSERT_TS":
            continue
        if c == "UPDATE_TS":
            update_parts.append(f"t.`{c}` = TIMESTAMP '{RUN_TS}'")
        else:
            update_parts.append(f"t.`{c}` = s.`{c}`")

    insert_cols = ", ".join([f"`{c}`" for c in target_columns])
    insert_vals = ", ".join([f"s.`{c}`" for c in target_columns])

    spark.sql(f"""
        MERGE INTO `{SILVER_SCHEMA}`.`{target_table}` t
        USING `{view_name}` s
          ON {on_expr}
        WHEN MATCHED AND {changed}
          THEN UPDATE SET {", ".join(update_parts)}
        WHEN NOT MATCHED
          THEN INSERT ({insert_cols})
               VALUES ({insert_vals})
    """)

    spark.catalog.dropTempView(view_name)

    metrics = (
        DeltaTable.forName(spark, target_name)
        .history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
        or {}
    )

    print(
        f"{target_table}: inserted="
        f"{metrics.get('numTargetRowsInserted', 0)}, "
        f"updated={metrics.get('numTargetRowsUpdated', 0)}"
    )


# =============================================================================
# DIMENSIONS
# =============================================================================

def load_dim_schedule():
    target = "SILVER_DIM_SCHEDULE"
    source = "Bronze.BRONZE_SCHEDULE"

    df = spark.sql(f"""
        SELECT
            CAST(SCHEDULE_ID AS BIGINT) AS SCHEDULE_ID,
            CAST(SCHEDULE_ID AS BIGINT) AS SCHEDULE_NK_ID,
            COALESCE(NULLIF(TRIM(SCHEDULE_DESC), ''), 'NOT AVAILABLE')
                AS SCHEDULE_DESC,
            COALESCE(
                TO_DATE(SCHEDULE_START_DATE),
                DATE '1900-01-01'
            ) AS SCHEDULE_START_DATE,
            COALESCE(
                TO_DATE(SCHEDULE_END_DATE),
                DATE '9999-12-31'
            ) AS SCHEDULE_END_DATE,
            CASE
                WHEN COALESCE(IS_ACTIVE, 1) = 1
                     AND COALESCE(IS_DELETED, 0) = 0
                THEN true
                ELSE false
            END AS IS_ACTIVE,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_SCHEDULE
    """)

    run_dq(
        df, target, source,
        "SCHEDULE_ID_NOT_NULL", "NOT_NULL",
        "SCHEDULE_ID must not be null", "ERROR",
        F.col("SCHEDULE_ID").isNull(),
        ["SCHEDULE_ID", "SCHEDULE_DESC"]
    )
    dq_unique(df, target, source, ["SCHEDULE_ID"])

    merge_type1(
        target, df,
        ["SCHEDULE_ID"],
        [
            "SCHEDULE_NK_ID","SCHEDULE_DESC","SCHEDULE_START_DATE",
            "SCHEDULE_END_DATE","IS_ACTIVE","IS_CURRENT",
            "EFFECTIVE_FROM_DATE","EFFECTIVE_TO_DATE"
        ],
    )


def load_dim_stop():
    target = "SILVER_DIM_STOP"
    source = "Bronze.BRONZE_STOP"

    df = spark.sql(f"""
        SELECT
            CAST(STOP_ID AS BIGINT) AS STOP_ID,
            CAST(STOP_ID AS BIGINT) AS STOP_NK_ID,
            NULLIF(TRIM(STOP_ABBR), '') AS STOP_ABBR,
            'NOT AVAILABLE' AS STOP_RIDE_ABBR,
            NULLIF(TRIM(STOP_NAME), '') AS STOP_NAME,
            NULLIF(TRIM(AT_STREET), '') AS AT_STREET,
            NULLIF(TRIM(ON_STREET), '') AS ON_STREET,
            NULLIF(TRIM(CAST(UNIT AS STRING)), '') AS UNIT,
            INITCAP(NULLIF(TRIM(CITY), '')) AS CITY,
            LPAD(CAST(ZIPCODE AS STRING), 5, '0') AS ZIPCODE,
            NULLIF(TRIM(ZONE), '') AS ZONE,
            CAST(STOP_LAT AS DOUBLE) AS STOP_LAT,
            CAST(STOP_LNG AS DOUBLE) AS STOP_LNG,
            CAST(NULL AS STRING) AS FACING_DIRECTION,
            COALESCE(NULLIF(UPPER(TRIM(STOP_TYPE)), ''), 'BUS') AS STOP_TYPE,
            CASE
                WHEN STATUS IS NULL THEN true
                WHEN UPPER(TRIM(STATUS)) = 'ACTIVE' THEN true
                ELSE false
            END AS IS_ACTIVE,
            COALESCE(
                CAST(IS_WHEELCHAIR_ACCESSIBLE AS BOOLEAN),
                false
            ) AS IS_WHEELCHAIR_ACCESSIBLE,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_STOP
    """)

    run_dq(
        df, target, source,
        "STOP_ID_NOT_NULL", "NOT_NULL",
        "STOP_ID must not be null", "ERROR",
        F.col("STOP_ID").isNull(),
        ["STOP_ID","STOP_NAME"]
    )
    dq_unique(df, target, source, ["STOP_ID"])
    run_dq(
        df, target, source,
        "STOP_LAT_RANGE", "RANGE",
        "Latitude must be between -90 and 90", "ERROR",
        F.col("STOP_LAT").isNotNull()
        & ~F.col("STOP_LAT").between(-90.0, 90.0),
        ["STOP_ID","STOP_LAT"]
    )
    run_dq(
        df, target, source,
        "STOP_LNG_RANGE", "RANGE",
        "Longitude must be between -180 and 180", "ERROR",
        F.col("STOP_LNG").isNotNull()
        & ~F.col("STOP_LNG").between(-180.0, 180.0),
        ["STOP_ID","STOP_LNG"]
    )
    run_dq(
        df, target, source,
        "ZIPCODE_LENGTH_5", "FORMAT",
        "ZIPCODE must be 5 characters after left padding", "WARNING",
        F.col("ZIPCODE").isNotNull() & (F.length("ZIPCODE") != 5),
        ["STOP_ID","ZIPCODE"]
    )

    merge_type1(
        target, df,
        ["STOP_ID"],
        [c for c in df.columns
         if c not in {"STOP_ID","READ_TS","INSERT_TS","UPDATE_TS"}],
    )


def load_dim_media_type():
    target = "SILVER_DIM_MEDIA_TYPE"
    source = "Bronze.BRONZE_MEDIA_TYPE"

    df = spark.sql(f"""
        SELECT
            CAST(MEDIA_TYPE_ID AS BIGINT) AS MEDIA_TYPE_ID,
            CAST(MEDIA_TYPE_ID AS BIGINT) AS MEDIA_TYPE_NK_ID,
            NULLIF(UPPER(TRIM(MEDIA_TYPE_ABBR)), '')
                AS MEDIA_TYPE_ABBR,
            COALESCE(NULLIF(TRIM(DESCRIPTION), ''), 'NOT AVAILABLE')
                AS MEDIA_TYPE_DESC,
            NULLIF(TRIM(CATEGORY), '') AS MEDIA_TYPE_CATEGORY,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_MEDIA_TYPE
    """)

    dq_unique(df, target, source, ["MEDIA_TYPE_ID"])

    merge_type1(
        target, df,
        ["MEDIA_TYPE_ID"],
        [c for c in df.columns
         if c not in {"MEDIA_TYPE_ID","READ_TS","INSERT_TS","UPDATE_TS"}],
    )


def load_dim_direction():
    target = "SILVER_DIM_DIRECTION"
    source = "Bronze.BRONZE_DIRECTION"

    df = spark.sql(f"""
        SELECT
            CAST(DIRECTION_ID AS BIGINT) AS DIRECTION_ID,
            CAST(DIRECTION_ID AS BIGINT) AS DIRECTION_NK_ID,
            NULLIF(UPPER(TRIM(DIR_SHORT)), '') AS DIRECTION_ABBR,
            CAST(NUMBER_NOTATION AS INT) AS DIRECTION_NUM,
            NULLIF(TRIM(DIRECTION_NAME), '') AS DIRECTION_NAME,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_DIRECTION
    """)

    dq_unique(df, target, source, ["DIRECTION_ID"])

    merge_type1(
        target, df,
        ["DIRECTION_ID"],
        [c for c in df.columns
         if c not in {"DIRECTION_ID","READ_TS","INSERT_TS","UPDATE_TS"}],
    )


def load_dim_block():
    target = "SILVER_DIM_BLOCK"
    source = "Bronze.BRONZE_BLOCK"

    df = spark.sql(f"""
        SELECT
            CAST(BLOCK_ID AS BIGINT) AS BLOCK_ID,
            CAST(BLOCK_ID AS BIGINT) AS BLOCK_NK_ID,
            CAST(SCHEDULE_ID AS BIGINT) AS SCHEDULE_ID,
            TRIM(BLOCK_NBR) AS BLOCK_NUMBER,
            NULLIF(TRIM(BLOCK_NAME), '') AS BLOCK_NAME,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_BLOCK
    """)

    run_dq(
        df, target, source,
        "BLOCK_NUMBER_NOT_NULL", "NOT_NULL",
        "BLOCK_NUMBER is required for Stop Activity fact lookup", "ERROR",
        F.col("BLOCK_NUMBER").isNull(),
        ["BLOCK_ID","SCHEDULE_ID","BLOCK_NUMBER"]
    )
    dq_unique(df, target, source, ["BLOCK_ID"])

    merge_type1(
        target, df,
        ["BLOCK_ID"],
        [c for c in df.columns
         if c not in {"BLOCK_ID","READ_TS","INSERT_TS","UPDATE_TS"}],
    )


def load_dim_vehicle():
    target = "SILVER_DIM_VEHICLE"
    source = "Bronze.BRONZE_VEHICLE"

    df = spark.sql(f"""
        SELECT
            CAST(VEHICLE_ID AS BIGINT) AS VEHICLE_ID,
            CAST(VEHICLE_ID AS BIGINT) AS VEHICLE_NK_ID,
            COALESCE(NULLIF(TRIM(MFG), ''), 'NOT AVAILABLE')
                AS VEHICLE_MAKE,
            UPPER(TRIM(TAG)) AS VEHICLE_TAG,
            'NOT AVAILABLE' AS VEHICLE_INET,
            COALESCE(NULLIF(TRIM(MFG_MODEL), ''), 'NOT AVAILABLE')
                AS VEHICLE_MODEL,
            COALESCE(NULLIF(UPPER(TRIM(VEHICLE_TYPE)), ''), 'NOT AVAILABLE')
                AS VEHICLE_TYPE_DESC,
            CAST(TOTAL_CAPACITY AS INT) AS VEHICLE_PSGR_CAPACITY,
            CAST(MODEL_YEAR AS INT) AS VEHICLE_YEAR,
            COALESCE(CAST(WHEEL_CHAIR_CAPACITY AS INT), 0)
                AS WHEELCHAIR_CAPACITY,
            CASE
                WHEN UPPER(TRIM(STATUS)) = 'ACTIVE' THEN true
                ELSE false
            END AS IS_ACTIVE,
            CAST(NULL AS DATE) AS IN_SERVICE_DATE,
            CAST(NULL AS DATE) AS RETIREMENT_DATE,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_VEHICLE
    """)

    run_dq(
        df, target, source,
        "VEHICLE_TAG_NOT_NULL", "NOT_NULL",
        "VEHICLE_TAG is required by mapping", "ERROR",
        F.col("VEHICLE_TAG").isNull()
        | (F.length(F.trim("VEHICLE_TAG")) == 0),
        ["VEHICLE_ID","VEHICLE_TAG"]
    )
    run_dq(
        df, target, source,
        "PASSENGER_CAPACITY_GT_ZERO", "BUSINESS_RULE",
        "Vehicle passenger capacity must be greater than zero", "ERROR",
        F.col("VEHICLE_PSGR_CAPACITY").isNull()
        | (F.col("VEHICLE_PSGR_CAPACITY") <= 0),
        ["VEHICLE_ID","VEHICLE_PSGR_CAPACITY"]
    )
    dq_unique(df, target, source, ["VEHICLE_ID"])

    merge_type1(
        target, df,
        ["VEHICLE_ID"],
        [c for c in df.columns
         if c not in {"VEHICLE_ID","READ_TS","INSERT_TS","UPDATE_TS"}],
    )


def load_dim_ridership_validation():
    target = "SILVER_DIM_RIDERSHIP_VALIDATION"
    source = "Bronze.BRONZE_STOP_ACTIVITY"

    # Mapping.xlsx explicitly says distinct source values.
    df = spark.sql(f"""
        SELECT DISTINCT
            CAST(RIDERSHIP_VALIDATION AS INT)
                AS RIDERSHIP_VALIDATION_ID,
            CASE CAST(RIDERSHIP_VALIDATION AS INT)
                WHEN 1 THEN 'VALID'
                WHEN 2 THEN 'INVALID'
                WHEN 3 THEN 'MISSING_DATA'
                WHEN 4 THEN 'MANUAL_REVIEW'
                WHEN 5 THEN 'OVERRIDE'
                WHEN 6 THEN 'PENDING'
                ELSE NULL
            END AS RIDERSHIP_VALIDATION_CATEGORY,
            CASE
                WHEN CAST(RIDERSHIP_VALIDATION AS INT) IN (1, 5)
                THEN true
                ELSE false
            END AS IS_VALID_RIDERSHIP,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_STOP_ACTIVITY
        WHERE RIDERSHIP_VALIDATION IS NOT NULL
    """)

    run_dq(
        df, target, source,
        "VALIDATION_CODE_DOMAIN", "DOMAIN",
        "Ridership validation code should be between 1 and 6", "WARNING",
        ~F.col("RIDERSHIP_VALIDATION_ID").isin(1,2,3,4,5,6),
        ["RIDERSHIP_VALIDATION_ID"]
    )
    dq_unique(df, target, source, ["RIDERSHIP_VALIDATION_ID"])

    merge_type1(
        target, df,
        ["RIDERSHIP_VALIDATION_ID"],
        ["RIDERSHIP_VALIDATION_CATEGORY","IS_VALID_RIDERSHIP"],
    )


def load_dim_route():
    target = "SILVER_DIM_ROUTE"
    source = "Bronze.BRONZE_ROUTE, Bronze.BRONZE_SCHEDULED_TRIP"

    # Based on supplied sample query:
    # ROUTE_SCHEDULE_ID = route_id + '_' + schedule_id
    # Use -1 when no schedule is available.
    df = spark.sql(f"""
        SELECT DISTINCT
            CONCAT_WS(
                '_',
                CAST(br.ROUTE_ID AS STRING),
                CAST(COALESCE(st.SCHEDULE_ID, -1) AS STRING)
            ) AS ROUTE_SCHEDULE_ID,
            CAST(br.ROUTE_ID AS BIGINT) AS ROUTE_ID,
            CAST(br.ROUTE_ID AS BIGINT) AS ROUTE_NK_ID,
            CAST(COALESCE(st.SCHEDULE_ID, -1) AS BIGINT)
                AS SCHEDULE_NK_ID,
            COALESCE(NULLIF(TRIM(br.ROUTE_NAME), ''), 'NOT AVAILABLE')
                AS ROUTE_NAME,
            COALESCE(NULLIF(TRIM(br.ROUTE_TYPE), ''), 'NOT AVAILABLE')
                AS ROUTE_TYPE,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_ROUTE br
        LEFT JOIN Bronze.BRONZE_SCHEDULED_TRIP st
          ON br.ROUTE_ID = st.ROUTE_ID
    """)

    run_dq(
        df, target, source,
        "ROUTE_ID_NOT_NULL", "NOT_NULL",
        "ROUTE_ID must not be null", "ERROR",
        F.col("ROUTE_ID").isNull(),
        ["ROUTE_SCHEDULE_ID","ROUTE_ID","SCHEDULE_NK_ID"]
    )
    dq_unique(df, target, source, ["ROUTE_SCHEDULE_ID"])

    merge_type1(
        target, df,
        ["ROUTE_SCHEDULE_ID"],
        [
            "ROUTE_ID","ROUTE_NK_ID","SCHEDULE_NK_ID","ROUTE_NAME",
            "ROUTE_TYPE","IS_CURRENT","EFFECTIVE_FROM_DATE",
            "EFFECTIVE_TO_DATE"
        ],
    )


def load_dim_trip():
    target = "SILVER_DIM_TRIP"
    source = "Bronze.BRONZE_TRIP, Bronze.BRONZE_SCHEDULED_TRIP"

    # Supplied sample query defines revenue trip as:
    # BT.STATUS = ACTIVE and scheduled-trip status is not CANCELLED.
    #
    # Scheduled_Trip can have many rows per trip (one per stop), so aggregate
    # cancellation status BEFORE joining; otherwise DISTINCT may still emit
    # conflicting IS_REVENUE_TRIP values for one TRIP_ID.
    df = spark.sql(f"""
        WITH scheduled_status AS (
            SELECT
                TRIP_ID,
                MAX(
                    CASE
                        WHEN UPPER(TRIM(TRIP_STATUS)) = 'CANCELLED'
                        THEN 1 ELSE 0
                    END
                ) AS HAS_CANCELLED_STOP
            FROM Bronze.BRONZE_SCHEDULED_TRIP
            GROUP BY TRIP_ID
        )
        SELECT
            CAST(bt.TRIP_ID AS BIGINT) AS TRIP_ID,
            CAST(bt.TRIP_ID AS BIGINT) AS TRIP_NK_ID,
            CAST(COALESCE(bt.SCHEDULE_ID, -1) AS BIGINT) AS SCHEDULE_ID,
            CASE
                WHEN UPPER(TRIM(bt.STATUS)) = 'ACTIVE'
                     AND COALESCE(ss.HAS_CANCELLED_STOP, 0) = 0
                THEN true
                ELSE false
            END AS IS_REVENUE_TRIP,
            NULLIF(TRIM(bt.TRIP_START_TIME), '') AS TRIP_START_TIME,
            NULLIF(TRIM(bt.TRIP_END_TIME), '') AS TRIP_END_TIME,
            NULLIF(TRIM(bt.TRIP_TYPE), '') AS TRIP_TYPE,
            CASE
                WHEN UPPER(TRIM(bt.STATUS)) = 'ACTIVE' THEN true
                ELSE false
            END AS IS_ACTIVE,
            true AS IS_CURRENT,
            DATE '1900-01-01' AS EFFECTIVE_FROM_DATE,
            DATE '9999-12-31' AS EFFECTIVE_TO_DATE,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM Bronze.BRONZE_TRIP bt
        LEFT JOIN scheduled_status ss
          ON bt.TRIP_ID = ss.TRIP_ID
    """)

    dq_unique(df, target, source, ["TRIP_ID"])

    merge_type1(
        target, df,
        ["TRIP_ID"],
        [
            "TRIP_NK_ID","SCHEDULE_ID","IS_REVENUE_TRIP",
            "TRIP_START_TIME","TRIP_END_TIME","TRIP_TYPE","IS_ACTIVE",
            "IS_CURRENT","EFFECTIVE_FROM_DATE","EFFECTIVE_TO_DATE"
        ],
    )


def load_dim_date():
    target = "SILVER_DIM_DATE"
    source = "Bronze.BRONZE_STOP_ACTIVITY"

    bounds = spark.sql("""
        SELECT
            MIN(TO_DATE(TRIP_DATE, 'dd-MM-yyyy')) AS MIN_DATE,
            MAX(TO_DATE(TRIP_DATE, 'dd-MM-yyyy')) AS MAX_DATE
        FROM Bronze.BRONZE_STOP_ACTIVITY
    """).first()

    min_date = bounds["MIN_DATE"]
    max_date = bounds["MAX_DATE"]

    if min_date is None or max_date is None:
        raise ValueError("No valid TRIP_DATE values available for DIM_DATE")

    df = spark.sql(f"""
        WITH d AS (
            SELECT EXPLODE(
                SEQUENCE(
                    DATE_SUB(DATE '{min_date}', 365),
                    DATE_ADD(DATE '{max_date}', 365),
                    INTERVAL 1 DAY
                )
            ) AS SERVICE_DATE
        ),
        c AS (
            SELECT
                SERVICE_DATE,
                CAST(DATE_FORMAT(SERVICE_DATE, 'yyyyMMdd') AS BIGINT)
                    AS DATE_ID,
                DATE_FORMAT(SERVICE_DATE, 'EEEE') AS SERVICE_DAY,
                CASE
                    WHEN DAYOFWEEK(SERVICE_DATE) IN (1,7)
                    THEN 'WEEKEND'
                    ELSE 'WEEKDAY'
                END AS SERVICE_DAY_TYPE,
                DATE_FORMAT(SERVICE_DATE, 'MMMM') AS MONTH_NAME,
                YEAR(SERVICE_DATE) AS CALENDER_YEAR,
                MONTH(SERVICE_DATE) AS CALENDER_MONTH_NUM,
                DATE_FORMAT(SERVICE_DATE, 'MMM yyyy')
                    AS MONTH_NAME_YEAR,
                WEEKOFYEAR(SERVICE_DATE) AS WEEK_NUM
            FROM d
        )
        SELECT
            SERVICE_DATE,
            DATE_ID,
            SERVICE_DAY,
            SERVICE_DAY_TYPE,
            MONTH_NAME,
            CASE
                WHEN {FISCAL_YEAR_START_MONTH} = 1
                THEN CALENDER_MONTH_NUM
                ELSE (
                    (CALENDER_MONTH_NUM - {FISCAL_YEAR_START_MONTH} + 12) % 12
                ) + 1
            END AS FISCAL_MONTH_NUM,
            CAST(
                CEIL(
                    (
                        CASE
                            WHEN {FISCAL_YEAR_START_MONTH} = 1
                            THEN CALENDER_MONTH_NUM
                            ELSE (
                                (
                                    CALENDER_MONTH_NUM
                                    - {FISCAL_YEAR_START_MONTH}
                                    + 12
                                ) % 12
                            ) + 1
                        END
                    ) / 3.0
                ) AS INT
            ) AS FISCAL_QUARTER,
            CASE
                WHEN {FISCAL_YEAR_START_MONTH} = 1
                THEN CALENDER_YEAR
                WHEN CALENDER_MONTH_NUM >= {FISCAL_YEAR_START_MONTH}
                THEN CALENDER_YEAR + 1
                ELSE CALENDER_YEAR
            END AS FISCAL_YEAR,
            CALENDER_YEAR,
            CALENDER_MONTH_NUM,
            MONTH_NAME_YEAR,
            WEEK_NUM,
            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS
        FROM c
    """)

    dq_unique(df, target, source, ["SERVICE_DATE"])

    merge_type1(
        target, df,
        ["SERVICE_DATE"],
        [c for c in df.columns
         if c not in {"SERVICE_DATE","READ_TS","INSERT_TS","UPDATE_TS"}],
    )



# FACT


def load_fact_ridership():
    target = "SILVER_FACT_RIDERSHIP"
    source = (
        "Bronze.BRONZE_STOP_ACTIVITY + Silver dimensions"
    )

    # Supplied sample query aggregates PSGR_ON / PSGR_MILES / PSGR_HRS.
    #
    # Important correction from the sample:
    # BSA.BLOCK contains values such as 1001-001, so it must match
    # DIM_BLOCK.BLOCK_NUMBER, not DIM_BLOCK.BLOCK_ID.
    #
    # Coordinates intentionally come from DIM_STOP, matching the sample query.
    stage = spark.sql(f"""
        SELECT
            COALESCE(dst.STOP_ID, -1) AS STOP_ID,
            COALESCE(ds.SCHEDULE_ID, -1) AS SCHEDULE_ID,
            COALESCE(db.BLOCK_ID, -1) AS BLOCK_ID,
            COALESCE(dmt.MEDIA_TYPE_ID, -1) AS MEDIA_TYPE_ID,
            COALESCE(dv.VEHICLE_ID, -1) AS VEHICLE_ID,
            COALESCE(dt.TRIP_ID, -1) AS TRIP_ID,
            COALESCE(drv.RIDERSHIP_VALIDATION_ID, -1)
                AS RIDERSHIP_VALIDATION_ID,
            COALESCE(dr.ROUTE_SCHEDULE_ID, '-1')
                AS ROUTE_SCHEDULE_ID,
            COALESCE(dr.ROUTE_ID, -1) AS ROUTE_ID,
            COALESCE(ddr.DIRECTION_ID, -1) AS DIRECTION_ID,

            CONCAT_WS(
                '_',
                COALESCE(ddr.DIRECTION_ABBR, 'NA'),
                CAST(COALESCE(dr.ROUTE_ID, -1) AS STRING),
                CAST(COALESCE(ds.SCHEDULE_ID, -1) AS STRING)
            ) AS DIR_ROUTE_SCHEDULE_ID,

            TO_DATE(bsa.TRIP_DATE, 'dd-MM-yyyy')
                AS SERVICE_DATE_OPERATED,
            TO_DATE(bsa.TRIP_DATE, 'dd-MM-yyyy')
                AS SERVICE_DATE,

            CAST(dst.STOP_LAT AS DOUBLE) AS LATITUDE,
            CAST(dst.STOP_LNG AS DOUBLE) AS LONGITUDE,

            SUM(COALESCE(CAST(bsa.PSGR_LOAD AS BIGINT),0)
            ) AS PSGR_ON_COUNT,
            CAST(
                SUM(COALESCE(CAST(bsa.PSGR_MILES AS DECIMAL(38,4)), 0))
                AS DECIMAL(38,4)
            ) AS PSGR_MILES,
            CAST(
                SUM(COALESCE(CAST(bsa.PSGR_HRS AS DECIMAL(38,4)), 0))
                AS DECIMAL(38,4)
            ) AS PSGR_HRS,

            TIMESTAMP '{RUN_TS}' AS READ_TS,
            TIMESTAMP '{RUN_TS}' AS INSERT_TS,
            CAST(NULL AS TIMESTAMP) AS UPDATE_TS

        FROM Bronze.BRONZE_STOP_ACTIVITY bsa

        LEFT JOIN silver.SILVER_DIM_ROUTE dr
          ON dr.ROUTE_SCHEDULE_ID = CONCAT_WS(
                '_',
                CAST(bsa.ROUTE AS STRING),
                CAST(COALESCE(bsa.SCHEDULE, -1) AS STRING)
             )

        LEFT JOIN silver.SILVER_DIM_TRIP dt
          ON dt.TRIP_ID = bsa.TRIP_ID

        LEFT JOIN silver.SILVER_DIM_BLOCK db
          ON db.BLOCK_NUMBER = TRIM(bsa.BLOCK)
         AND db.SCHEDULE_ID = bsa.SCHEDULE

        LEFT JOIN silver.SILVER_DIM_DATE dd
          ON dd.SERVICE_DATE = TO_DATE(
                bsa.TRIP_DATE,
                'dd-MM-yyyy'
             )

        LEFT JOIN silver.SILVER_DIM_DIRECTION ddr
          ON ddr.DIRECTION_ID = bsa.DIRECTION

        LEFT JOIN silver.SILVER_DIM_MEDIA_TYPE dmt
          ON dmt.MEDIA_TYPE_ID = bsa.MEDIA_TYPE

        LEFT JOIN silver.SILVER_DIM_RIDERSHIP_VALIDATION drv
          ON drv.RIDERSHIP_VALIDATION_ID =
             bsa.RIDERSHIP_VALIDATION

        LEFT JOIN silver.SILVER_DIM_SCHEDULE ds
          ON ds.SCHEDULE_ID = bsa.SCHEDULE

        LEFT JOIN silver.SILVER_DIM_STOP dst
          ON dst.STOP_ID = bsa.STOP_ID

        LEFT JOIN silver.SILVER_DIM_VEHICLE dv
          ON dv.VEHICLE_ID = bsa.VEHICLE

        GROUP BY
            COALESCE(dst.STOP_ID, -1),
            COALESCE(ds.SCHEDULE_ID, -1),
            COALESCE(db.BLOCK_ID, -1),
            COALESCE(dmt.MEDIA_TYPE_ID, -1),
            COALESCE(dv.VEHICLE_ID, -1),
            COALESCE(dt.TRIP_ID, -1),
            COALESCE(drv.RIDERSHIP_VALIDATION_ID, -1),
            COALESCE(dr.ROUTE_SCHEDULE_ID, '-1'),
            COALESCE(dr.ROUTE_ID, -1),
            COALESCE(ddr.DIRECTION_ID, -1),
            COALESCE(ddr.DIRECTION_ABBR, 'NA'),
            TO_DATE(bsa.TRIP_DATE, 'dd-MM-yyyy'),
            CAST(dst.STOP_LAT AS DOUBLE),
            CAST(dst.STOP_LNG AS DOUBLE)
    """)

    # Deterministic fact key generated AFTER aggregation.
    key_terms = [
        F.coalesce(F.col("STOP_ID").cast("string"), F.lit("-1")),
        F.coalesce(F.col("SCHEDULE_ID").cast("string"), F.lit("-1")),
        F.coalesce(F.col("BLOCK_ID").cast("string"), F.lit("-1")),
        F.coalesce(F.col("MEDIA_TYPE_ID").cast("string"), F.lit("-1")),
        F.coalesce(F.col("VEHICLE_ID").cast("string"), F.lit("-1")),
        F.coalesce(F.col("TRIP_ID").cast("string"), F.lit("-1")),
        F.coalesce(
            F.col("RIDERSHIP_VALIDATION_ID").cast("string"),
            F.lit("-1")
        ),
        F.coalesce(F.col("ROUTE_SCHEDULE_ID"), F.lit("-1")),
        F.coalesce(F.col("DIRECTION_ID").cast("string"), F.lit("-1")),
        F.coalesce(
            F.date_format("SERVICE_DATE", "yyyy-MM-dd"),
            F.lit("1900-01-01")
        ),
        F.coalesce(F.col("LATITUDE").cast("string"), F.lit("__null__")),
        F.coalesce(F.col("LONGITUDE").cast("string"), F.lit("__null__")),
    ]

    df = stage.withColumn(
        "FACT_RIDERSHIP_ID",
        F.xxhash64(*key_terms)
    ).select(
        "FACT_RIDERSHIP_ID",
        "STOP_ID",
        "SCHEDULE_ID",
        "BLOCK_ID",
        "MEDIA_TYPE_ID",
        "VEHICLE_ID",
        "TRIP_ID",
        "RIDERSHIP_VALIDATION_ID",
        "ROUTE_SCHEDULE_ID",
        "ROUTE_ID",
        "DIRECTION_ID",
        "DIR_ROUTE_SCHEDULE_ID",
        "SERVICE_DATE_OPERATED",
        "SERVICE_DATE",
        "LATITUDE",
        "LONGITUDE",
        "PSGR_ON_COUNT",
        "PSGR_MILES",
        "PSGR_HRS",
        "READ_TS",
        "INSERT_TS",
        "UPDATE_TS",
    )

  
    # FACT DQ
    # - Unknown dimension (-1) is logged as WARNING, not load failure.
    # - Broken date and coordinates are ERROR because they break the fact grain.
   

    run_dq(
        df, target, source,
        "SERVICE_DATE_NOT_NULL", "NOT_NULL",
        "Fact service date must parse successfully", "ERROR",
        F.col("SERVICE_DATE").isNull(),
        ["TRIP_ID","SERVICE_DATE"]
    )

    run_dq(
        df, target, source,
        "STOP_LOOKUP_RESOLVED", "REFERENTIAL_INTEGRITY",
        "STOP_ID should resolve to Silver stop dimension", "WARNING",
        F.col("STOP_ID") == -1,
        ["TRIP_ID","STOP_ID","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "SCHEDULE_LOOKUP_RESOLVED", "REFERENTIAL_INTEGRITY",
        "SCHEDULE_ID should resolve to Silver schedule dimension", "WARNING",
        F.col("SCHEDULE_ID") == -1,
        ["TRIP_ID","SCHEDULE_ID","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "BLOCK_LOOKUP_RESOLVED", "REFERENTIAL_INTEGRITY",
        "BLOCK_NUMBER + SCHEDULE should resolve to Silver block dimension",
        "WARNING",
        F.col("BLOCK_ID") == -1,
        ["TRIP_ID","BLOCK_ID","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "VEHICLE_LOOKUP_RESOLVED", "REFERENTIAL_INTEGRITY",
        "VEHICLE_ID should resolve to Silver vehicle dimension", "WARNING",
        F.col("VEHICLE_ID") == -1,
        ["TRIP_ID","VEHICLE_ID","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "ROUTE_SCHEDULE_LOOKUP_RESOLVED", "REFERENTIAL_INTEGRITY",
        "Route + schedule should resolve to Silver route dimension", "WARNING",
        F.col("ROUTE_SCHEDULE_ID") == "-1",
        ["TRIP_ID","ROUTE_ID","ROUTE_SCHEDULE_ID","SERVICE_DATE"]
    )

    run_dq(
        df, target, source,
        "LATITUDE_RANGE", "RANGE",
        "Fact latitude must be between -90 and 90", "ERROR",
        F.col("LATITUDE").isNotNull()
        & ~F.col("LATITUDE").between(-90.0, 90.0),
        ["STOP_ID","LATITUDE"]
    )
    run_dq(
        df, target, source,
        "LONGITUDE_RANGE", "RANGE",
        "Fact longitude must be between -180 and 180", "ERROR",
        F.col("LONGITUDE").isNotNull()
        & ~F.col("LONGITUDE").between(-180.0, 180.0),
        ["STOP_ID","LONGITUDE"]
    )

    run_dq(
        df, target, source,
        "PSGR_ON_NON_NEGATIVE", "BUSINESS_RULE",
        "Aggregated passenger-on count must not be negative", "ERROR",
        F.col("PSGR_ON_COUNT") < 0,
        ["TRIP_ID","STOP_ID","PSGR_ON_COUNT","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "PSGR_MILES_NON_NEGATIVE", "BUSINESS_RULE",
        "Passenger miles must not be negative", "ERROR",
        F.col("PSGR_MILES") < 0,
        ["TRIP_ID","STOP_ID","PSGR_MILES","SERVICE_DATE"]
    )
    run_dq(
        df, target, source,
        "PSGR_HRS_NON_NEGATIVE", "BUSINESS_RULE",
        "Passenger hours must not be negative", "ERROR",
        F.col("PSGR_HRS") < 0,
        ["TRIP_ID","STOP_ID","PSGR_HRS","SERVICE_DATE"]
    )

    dq_unique(
        df,
        target,
        source,
        ["FACT_RIDERSHIP_ID"],
        severity="ERROR"
    )

    merge_type1(
        target,
        df,
        ["FACT_RIDERSHIP_ID"],
        [
            c for c in df.columns
            if c not in {
                "FACT_RIDERSHIP_ID",
                "READ_TS",
                "INSERT_TS",
                "UPDATE_TS"
            }
        ],
    )



# =============================================================================
# METADATA-DRIVEN DRIVER - DIMENSIONS FIRST, FACT LAST
# =============================================================================

# Existing transformation functions are intentionally retained.
# PARAM_1 in admin_ops.admin_load_control determines which function executes.
LOADERS = {
    "load_dim_schedule": load_dim_schedule,
    "load_dim_stop": load_dim_stop,
    "load_dim_media_type": load_dim_media_type,
    "load_dim_direction": load_dim_direction,
    "load_dim_block": load_dim_block,
    "load_dim_vehicle": load_dim_vehicle,
    "load_dim_ridership_validation": load_dim_ridership_validation,
    "load_dim_route": load_dim_route,
    "load_dim_trip": load_dim_trip,
    "load_dim_date": load_dim_date,
    "load_fact_ridership": load_fact_ridership,
}

p_target_table_clean = (p_target_table or "").strip()
p_target_table_escaped = p_target_table_clean.replace("'", "''")

target_filter_clause = ""

if p_target_table_clean:
    target_filter_clause = f"""
      AND (
            LOWER(DESTINATION_TABLE_NAME) = LOWER('{p_target_table_escaped}')
         OR LOWER(PARAM_1) = LOWER('{p_target_table_escaped}')
         OR CAST(ROW_ID AS STRING) = '{p_target_table_escaped}'
      )
    """

# IMPORTANT:
# Use an f-string AND the literal fully-qualified admin_ops table name.
# This prevents Spark from ever receiving:
#   FROM admin_ops.admin_load_control
control_sql = f"""
    SELECT
        ROW_ID,
        TRANSFER_TYPE,
        SOURCE_SYSTEM,
        SOURCE_SCHEMA,
        SOURCE_TABLE_NAME,
        DESTINATION_SCHEMA,
        DESTINATION_TABLE_NAME,
        LOAD_TYPE,
        PRIMARY_KEY,
        INCREMENTAL_COLUMN,
        EXECUTION_ORDER,
        JOB_DEPENDENCIES,
        RUN_ACTIVE_FLAG,
        PARAM_1,
        PARAM_2

    FROM admin_ops.admin_load_control

    WHERE RUN_ACTIVE_FLAG = 1
      AND UPPER(TRANSFER_TYPE) = 'BRZ_TO_SIL'

      {target_filter_clause}

    ORDER BY
        EXECUTION_ORDER,
        ROW_ID
"""

print("Silver metadata query:")
print(control_sql)

control_df = spark.sql(control_sql)

control_rows = control_df.collect()

print(f"Silver jobs selected: {len(control_rows)}")
display(control_df)

if not control_rows:
    raise ValueError(
        "No active BRZ_TO_SIL metadata rows were found in "
        f"{CONTROL_TABLE} for p_target_table='{p_target_table_clean}'."
    )

print(f"Metadata jobs selected: {len(control_rows)}")
for r in control_rows:
    print(
        f"  order={r['EXECUTION_ORDER']} | "
        f"row_id={r['ROW_ID']} | "
        f"{r['SOURCE_TABLE_NAME']} -> {r['DESTINATION_TABLE_NAME']} | "
        f"loader={r['PARAM_1']}"
    )

for job in control_rows:
    job_dict = job.asDict(recursive=True)

    row_id = job_dict.get("ROW_ID")
    source_schema = (job_dict.get("SOURCE_SCHEMA") or BRONZE_SCHEMA).strip()
    source_table_name = (job_dict.get("SOURCE_TABLE_NAME") or "").strip()
    destination_schema = (job_dict.get("DESTINATION_SCHEMA") or SILVER_SCHEMA).strip()
    destination_table_name = (job_dict.get("DESTINATION_TABLE_NAME") or "").strip()
    loader_name = (job_dict.get("PARAM_1") or "").strip()
    execution_order = job_dict.get("EXECUTION_ORDER")
    dependencies = job_dict.get("JOB_DEPENDENCIES")
    primary_key = job_dict.get("PRIMARY_KEY")

    if not destination_table_name:
        raise ValueError(f"ROW_ID {row_id}: DESTINATION_TABLE_NAME is required.")

    if not loader_name:
        raise ValueError(
            f"ROW_ID {row_id} / {destination_table_name}: "
            "PARAM_1 must contain the Silver loader function name."
        )

    loader = LOADERS.get(loader_name)
    if loader is None:
        raise ValueError(
            f"ROW_ID {row_id} / {destination_table_name}: "
            f"Unknown loader '{loader_name}'. Valid loaders: {sorted(LOADERS)}"
        )

    source_full = (
        f"{source_schema}.{source_table_name}"
        if source_table_name
        else source_schema
    )
    destination_full = f"{destination_schema}.{destination_table_name}"
    job_name = f"NB_SILVER_TRANSFORM:{destination_table_name}"
    job_start_ts = datetime.now(timezone.utc)

    print("\n" + "=" * 90)
    print(
        f"Loading {destination_table_name} | "
        f"metadata row={row_id} | order={execution_order} | loader={loader_name}"
    )
    if dependencies:
        print(f"Dependencies: {dependencies}")
    print("=" * 90)

    try:
        loader()

        target_name = f"{SILVER_SCHEMA}.{destination_table_name}"
        total_rows = (
            spark.table(target_name).count()
            if spark.catalog.tableExists(target_name)
            else 0
        )

        information = (
            f"Metadata-driven BRZ_TO_SIL load completed. "
            f"loader={loader_name}; execution_order={execution_order}; "
            f"primary_key={primary_key}; dependencies={dependencies}"
        )

        write_processing_log(
            job_name=job_name,
            job_start_ts=job_start_ts,
            job_status="SUCCESS",
            source_table=source_full,
            destination_table=destination_full,
            total_rows=total_rows,
            information=information,
            processed_condition=f"ROW_ID={row_id}; RUN_ACTIVE_FLAG=1",
        )

        print(
            f"SUCCESS: {destination_table_name} | "
            f"target row count={total_rows:,}"
        )

    except Exception as exc:
        error_message = str(exc)[:4000]

        write_processing_log(
            job_name=job_name,
            job_start_ts=job_start_ts,
            job_status="FAILED",
            source_table=source_full,
            destination_table=destination_full,
            total_rows=0,
            information=(
                f"Metadata-driven BRZ_TO_SIL load failed. "
                f"loader={loader_name}; execution_order={execution_order}"
            ),
            error_id=type(exc).__name__,
            error_message=error_message,
            processed_condition=f"ROW_ID={row_id}; RUN_ACTIVE_FLAG=1",
        )

        print(f"FAILED: {destination_table_name}: {error_message}")
        raise

print(
    f"\nSilver transformation run complete. "
    f"execution_id={EXECUTION_ID}. "
    f"Review {PROCESSING_LOG_TABLE} for processing status and "
    f"{DQ_TABLE} for DQ results."
)

